EMAIL SPAM DETECTION

extract subject + body from emails, create tf idf index using spam and ham, then create numerical(tf idf) representations of the data

In [7]:
from pathlib import Path
import re
import string
from email import policy
from email.parser import BytesParser

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 1: paths
base_dir = Path("/Users/aryamanwade/Desktop/ml_prac/ch2")
ham_dir = base_dir / "ham"
spam_dir = base_dir / "spam"
out_dir = base_dir / "q4_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

# Step 2: cleaning rules
url_pattern = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
number_pattern = re.compile(r"\b\d+(?:[\.,]\d+)?\b")
punct_table = str.maketrans("", "", string.punctuation)


def extract_subject_body(file_path: Path) -> str:
    # read one email
    with open(file_path, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    subject = msg.get("Subject", "")
    body_parts = []

    # get text body from plain/html parts
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() in {"text/plain", "text/html"}:
                try:
                    body_parts.append(part.get_content())
                except Exception:
                    payload = part.get_payload(decode=True)
                    if payload:
                        body_parts.append(payload.decode("utf-8", errors="ignore"))
    else:
        try:
            body_parts.append(msg.get_content())
        except Exception:
            payload = msg.get_payload(decode=True)
            if payload:
                body_parts.append(payload.decode("utf-8", errors="ignore"))

    body = "\n".join(p for p in body_parts if isinstance(p, str))
    return f"{subject}\n{body}".strip()


def clean_text(text: str) -> str:
    # lowercase, replace url/number, remove punctuation
    text = text.lower()
    text = url_pattern.sub(" url ", text)
    text = number_pattern.sub(" number ", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_folder(folder: Path):
    # ignore cmds file
    texts = []
    for fp in sorted(folder.iterdir()):
        if not fp.is_file():
            continue
        if fp.name.lower() == "cmds":
            continue
        texts.append(clean_text(extract_subject_body(fp)))
    return texts

# Step 3: load data
ham_texts = load_folder(ham_dir)
spam_texts = load_folder(spam_dir)

print("ham emails:", len(ham_texts))
print("spam emails:", len(spam_texts))

# Step 4: build shared tf-idf index
all_texts = ham_texts + spam_texts
vectorizer = TfidfVectorizer()
X_all = vectorizer.fit_transform(all_texts)

X_ham = X_all[:len(ham_texts)]
X_spam = X_all[len(ham_texts):]

print("features:", X_all.shape[1])
print("ham matrix:", X_ham.shape)
print("spam matrix:", X_spam.shape)

# Step 5: save separate files
ham_out = out_dir / "ham_tfidf.npz"
spam_out = out_dir / "spam_tfidf.npz"

sparse.save_npz(ham_out, X_ham)
sparse.save_npz(spam_out, X_spam)

print("saved:", ham_out)
print("saved:", spam_out)

ham emails: 1399
spam emails: 500
features: 37550
ham matrix: (1399, 37550)
spam matrix: (500, 37550)
saved: /Users/aryamanwade/Desktop/ml_prac/ch2/q4_outputs/ham_tfidf.npz
saved: /Users/aryamanwade/Desktop/ml_prac/ch2/q4_outputs/spam_tfidf.npz


load npz(numpy archive) and add tags 0 (ham) 1(Spam) and create training split

In [8]:
from scipy import sparse
import numpy as np
ham_out = out_dir / "ham_tfidf.npz"
spam_out = out_dir / "spam_tfidf.npz"
X_ham = sparse.load_npz(ham_out)
X_spam = sparse.load_npz(spam_out)
X = sparse.vstack([X_ham, X_spam], format="csr")
y = np.r_[np.zeros(X_ham.shape[0], dtype=int), np.ones(X_spam.shape[0], dtype=int)]
print(X.shape, y.shape)

(1899, 37550) (1899,)


In [9]:
from sklearn.model_selection import train_test_split

# Step 6: stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:", {0: int((y_train == 0).sum()), 1: int((y_train == 1).sum())})
print("y_test distribution:", {0: int((y_test == 0).sum()), 1: int((y_test == 1).sum())})

X_train: (1519, 37550)
X_test: (380, 37550)
y_train distribution: {0: 1119, 1: 400}
y_test distribution: {0: 280, 1: 100}


Linear Regression VS SVM

Logistic Regression
→ optimizes probabilities (log loss)
→ focuses on fitting all points well
SVM
→ optimizes margin (distance between classes)
→ focuses on separating the boundary as confidently as possible

Linear Regression

Logistic regression works well here because TF-IDF converts each email into a high-dimensional vector where each word represents a feature. The model learns a weight for each word, indicating how strongly it signals spam or ham. It then computes a weighted sum of these features to produce a score, which is converted into a probability. Since spam classification largely depends on the presence of certain keywords, this linear combination of word signals is highly effective.

In [10]:
from sklearn.metrics import precision_score, recall_score;

Hyper param tuning to balance recall and precision

1. Threholds: reduce to increase recall

2. class_weight, using balanced to weigh spams(1) higher. it makes the loss function peanlise errors on spam(1) more than 0. 

3. penalty, and C. L2 prevents overfitting byt adding a penalty to loss sum of squared weights to loss function. Small C strong regularisation. L2 makes the weights small, L1 shrinks to 0, Elastic net (mix of both)



In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 7: logistic regression with spam-aware class weighting
log_reg = LogisticRegression(penalty="l1", C=0.7, solver="liblinear", class_weight="balanced")
log_reg.fit(X_train, y_train)

# Use custom decision threshold
y_train_proba = log_reg.predict_proba(X_train)[:, 1]
y_test_proba = log_reg.predict_proba(X_test)[:, 1]
y_train_pred = (y_train_proba >= 0.5).astype(int)
y_test_pred = (y_test_proba >= 0.5).astype(int)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}, Recall: {recall_score(y_test, y_test_pred):.4f}")

Training accuracy: 0.9645
Test accuracy: 0.9395
Precision: 0.9053, Recall: 0.8600


/Users/aryamanwade/Desktop/ml_prac/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/aryamanwade/Desktop/ml_prac/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Linear SVM finds the hyperplane that best separates spam and ham by maximizing the margin between them.


TF-IDF creates a very high-dimensional space


In high dimensions, data is often (almost) linearly separable


Spam vs ham differs strongly in word usage → clear separation


Hyper param tuning to balance precision and recall

1. C margin size vs classification errors.

Large C → tries very hard to classify training points correctly
→ tighter margin → more conservative → high precision, lower recall

With lower C, we’re okay misclassifying some points in exchange for a larger margin.


2. class_weight: without this all mistakes cost the same, we can weigh missclassifying as 2x importance. only loss is increased for wrong classificaitons.

3. Loss function: the seperation distance we are trying to maximise. we need to control margin and add penalty for misclassification. High C adds high loss for mis classification, class_weight can add varibale loss for respective miss classifications.





In [41]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# Step 8: baseline linear SVM
lin_svm = LinearSVC(C=0.7, class_weight="balanced")
lin_svm.fit(X_train, y_train)

y_train_pred_svm = lin_svm.predict(X_train)
y_test_pred_svm = lin_svm.predict(X_test)

train_acc_svm = accuracy_score(y_train, y_train_pred_svm)
test_acc_svm = accuracy_score(y_test, y_test_pred_svm)

print(f"Linear SVM training accuracy: {train_acc_svm:.4f}")
print(f"Linear SVM test accuracy: {test_acc_svm:.4f}")

print(f"Precision: {precision_score(y_test, y_test_pred_svm):.4f}, Recall: {recall_score(y_test, y_test_pred_svm):.4f}")

Linear SVM training accuracy: 1.0000
Linear SVM test accuracy: 0.9711
Precision: 0.9684, Recall: 0.9200
